# LoRA: адаптер, градиенты и merge

**LoRA: Low-Rank Adaptation of Large Language Models** · Hu et al. · 2021

[Оригинальная статья](https://arxiv.org/abs/2106.09685)

**Цель урока:** Создать низкоранговое обновление, обучить его и проверить эквивалентность merged весов.

## Перед началом

Основы обучения и умножение матриц; вводный урок LoRA.

Около 40–60 минут с заданиями. CPU, PyTorch и Matplotlib; без скачивания весов и датасетов. Время — ориентир для чтения и работы, а не benchmark выполнения.

## Задача и идея

При fine-tuning не обязательно менять каждую матрицу целиком. LoRA ограничивает обновление низким рангом, сохраняя базовые веса замороженными. Это ограничение класса обновлений, а не просто сжатие уже обученной матрицы.

Две меньшие матрицы экономят trainable parameters и optimizer states. Но память базовой модели и активаций остаётся. В этой работе разбираем один Linear; подбор target modules в LLM и качество downstream задач — следующий уровень эксперимента.

## Граница воспроизведения

Один Linear без dropout/quantization. Не QLoRA, не PEFT integration и не воспроизведение benchmark статьи.

Код написан для этого урока по математическим идеям статьи. Это не скопированный авторский репозиторий. Успех assertions подтверждает указанные свойства реализации, но не результаты статьи.

## Формула → тензоры

`y = x W0^T + (alpha/r) x A^T B^T`, где `A:[r,in]`, `B:[out,r]`. Trainable count `r*(in+out)`.

При B=0 начальный output совпадает с базой. В первый backward A обычно получает нулевой градиент, а B — ненулевой; это ожидаемо. Если обе матрицы равны нулю, оба градиента исчезают. Merge: `Wmerged = W0 + (alpha/r) B A`.

In [ ]:
%matplotlib inline
import math
import torch
from torch import nn
from torch.nn import functional as F
import matplotlib.pyplot as plt
torch.set_num_threads(1)
torch.manual_seed(42)
plt.rcParams.update({'figure.figsize': (7, 3), 'axes.spines.top': False, 'axes.spines.right': False})
print('PyTorch:', torch.__version__, '| CPU | seed=42')

## Шаг 1

Создаём слой и проверяем исходную функцию и асимметрию первых градиентов.

In [ ]:
class LoRALinear(nn.Module):
    def __init__(self, inputs=16, outputs=12, rank=3, alpha=6):
        super().__init__()
        self.base = nn.Linear(inputs, outputs, bias=False).requires_grad_(False)
        self.A = nn.Parameter(torch.randn(rank, inputs) * 0.02)
        self.B = nn.Parameter(torch.zeros(outputs, rank))
        self.scale = alpha / rank
    def forward(self, x):
        return self.base(x) + self.scale * (x @ self.A.T @ self.B.T)
layer = LoRALinear(); x = torch.randn(64, 16)
target = torch.randn(64, 12)
assert torch.equal(layer(x), layer.base(x))
F.mse_loss(layer(x), target).backward()
print('first A/B grad:', layer.A.grad.norm().item(), layer.B.grad.norm().item())
assert layer.A.grad.norm() == 0 and layer.B.grad.norm() > 0
assert layer.base.weight.grad is None

## Шаг 2

Обучаем только адаптер на контролируемом low-rank изменении. Target специально имеет низкий ранг — это проверка реализации, а не сравнение с full tuning.

In [ ]:
frozen = layer.base.weight.detach().clone()
target_update = torch.randn(12, 3) @ torch.randn(3, 16) * 0.1
target = F.linear(x, frozen + target_update)
optimizer = torch.optim.AdamW([layer.A, layer.B], lr=0.03, weight_decay=0)
losses = []
for step in range(160):
    optimizer.zero_grad(set_to_none=True)
    loss = F.mse_loss(layer(x), target); loss.backward(); optimizer.step()
    losses.append(loss.item())
assert torch.equal(layer.base.weight, frozen)
assert losses[-1] < losses[0]
print('trainable:', layer.A.numel() + layer.B.numel(), 'full matrix:', frozen.numel())
plt.plot(losses, color='#d73535'); plt.xlabel('step'); plt.ylabel('adapter MSE'); plt.show()

## Шаг 3

Merge должен сохранять функцию. После merge нельзя повторно добавлять адаптер к уже изменённой базе.

In [ ]:
with torch.no_grad():
    merged_weight = layer.base.weight + layer.scale * (layer.B @ layer.A)
    adapter_output = layer(x)
    merged_output = F.linear(x, merged_weight)
error = (adapter_output - merged_output).abs().max().item()
print('merge error:', error)
assert torch.allclose(adapter_output, merged_output, atol=1e-5)
bad_A = nn.Parameter(torch.zeros(3, 16)); bad_B = nn.Parameter(torch.zeros(12, 3))
(x @ bad_A.T @ bad_B.T - target).square().mean().backward()
assert bad_A.grad.norm() == 0 and bad_B.grad.norm() == 0

## Как читать результат

Снижение loss здесь ожидаемо: target построен как rank-3 update. Проверка замороженной базы выявляет случайное full tuning. Merge сравнивается с tolerance, поскольку порядок операций меняет округление float32.

## Самостоятельные эксперименты

1. Поставьте rank=1 при rank-3 target.
2. Увеличьте alpha, оставив learning rate прежним.
3. Выполните merge дважды и сравните outputs.

Сначала запишите прогноз, затем меняйте по одному фактору. Для честного сравнения сбрасывайте ядро и выполняйте все ячейки с начала. Сохраните копию для своих изменений.

## Проверка рассуждений — после эксперимента

Rank-1 адаптер в общем случае не представит rank-3 update. Alpha масштабирует эффективное обновление. Повторный merge удваивает delta и меняет функцию: merge должен быть управляемой операцией состояния.

## Что записать в лабораторный журнал

- Гипотеза и изменённый параметр.
- Seed, размер данных, число шагов и используемая precision.
- Значения проверок до и после, график и объяснение расхождения с прогнозом.
- Что этот эксперимент **не** позволяет заключить.

Связанный раздел доступен по кнопке «Открыть раздел студии» над уроком.